# Lesson 22: Regularization
**Module 4: ML Regression**

Interactive notebook for Ridge, Lasso, and ElasticNet.

## Learning Objectives
- Understand regularization penalties
- Apply Ridge, Lasso, and ElasticNet
- Tune hyperparameters with cross-validation
- Compare feature selection effects

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
np.random.seed(42)

# Generate sample data with some irrelevant features
n = 200
X = np.random.randn(n, 10)
true_coef = np.array([3, 2, 1, 0, 0, 0, 0, 0, 0, 0])  # Only 3 relevant
y = X @ true_coef + np.random.randn(n) * 0.5

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f'Training samples: {len(X_train)}, Test samples: {len(X_test)}')

## 1. Scale Features First

In [ ]:
# Always scale before regularization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Features scaled to mean=0, std=1')

## 2. Compare Regularization Methods

In [ ]:
# Fit different models
ols = LinearRegression().fit(X_train_scaled, y_train)
ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
lasso = Lasso(alpha=0.1).fit(X_train_scaled, y_train)

# Compare coefficients
coef_df = pd.DataFrame({
    'True': true_coef,
    'OLS': ols.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_
}, index=[f'X{i}' for i in range(10)])
print(coef_df.round(3))

In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(10)
width = 0.2
ax.bar(x - width, ols.coef_, width, label='OLS')
ax.bar(x, ridge.coef_, width, label='Ridge')
ax.bar(x + width, lasso.coef_, width, label='Lasso')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
ax.set_title('Coefficient Comparison')
ax.legend()
plt.show()

## 3. Cross-Validation for Alpha

In [ ]:
# Test different alpha values
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    scores = cross_val_score(lasso, X_train_scaled, y_train, cv=5, scoring='r2')
    n_nonzero = np.sum(lasso.fit(X_train_scaled, y_train).coef_ != 0)
    print(f'alpha={alpha:5.3f}: R2={scores.mean():.3f} (+/- {scores.std():.3f}), features={n_nonzero}')

## Exercise: Find Optimal Regularization
Use GridSearchCV to find the best alpha.

In [ ]:
# Exercise: Use GridSearchCV to find optimal alpha for Lasso
# 1. Import GridSearchCV
# 2. Define param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0]}
# 3. Fit and find best_params_

# YOUR CODE HERE

---
## Solution
<details>
<summary>Click to reveal</summary>

```python
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0]}
grid = GridSearchCV(Lasso(), param_grid, cv=5, scoring='r2')
grid.fit(X_train_scaled, y_train)

print(f'Best alpha: {grid.best_params_}')
print(f'Best R2: {grid.best_score_:.3f}')
print(f'Non-zero coefficients: {np.sum(grid.best_estimator_.coef_ != 0)}')
```
</details>